# Notebook 06 — RB Molecular Subtype Scoring

**Project:** RetinoblastomaAtlas  
**Input:** `data/processed/06_cnv.h5ad`  
**Output:** `data/processed/07_subtyped.h5ad`, subtype UMAPs, violin plots, composition bar chart

---

## Scientific Background

Liu et al. (2021, *Nature Communications*) defined two molecular subtypes of retinoblastoma tumour cells using single-cell transcriptomics:

### Subtype 1 — Mature cone-like
- Expresses cone photoreceptor maturation markers: **ARR3, RXRG, OPN1SW, THRB, GNAT2**
- Resembles the post-mitotic cone precursor cell of origin (Singh et al. 2018, *PNAS*)
- More common in heritable (germline RB1 mutation) cases
- Better-differentiated; lower proliferative index
- Lower tendency for vitreous seeding or optic nerve invasion

### Subtype 2 — Dedifferentiated / stemness
- Expresses mesenchymal/stemness markers: **VIM, NES, CD44, SOX2, PROM1, FN1**
- Associated with MYCN amplification (chr2p)
- Higher metastatic risk — extraocular extension
- Present in both germline and somatic cases but enriched in high-risk patients

## Significance in This Atlas

By scoring every cell with Subtype 1/2 signatures, we can:
1. Test whether **extraocular tumour cells are enriched for Subtype 2** (supporting the intraocular → extraocular dedifferentiation model)
2. Trace subtype composition along the cone precursor trajectory (RNA velocity, notebook 07)
3. Correlate subtype scores with TGF-β activity (notebook 10)

## Method — `sc.tl.score_genes()`

Scanpy's `score_genes()` implements the **Tirosh et al. 2016** modified z-score method:
- For each cell, compute the mean expression of signature genes
- Subtract the mean of a random **control gene set from the same expression bins**
- Result is robust to library-size effects

**References:**  
- Liu J et al. *Nat Commun* 2021. https://doi.org/10.1038/s41467-021-25595-x  
- Singh HP et al. *PNAS* 2018. https://doi.org/10.1073/pnas.1808903115  
- Wan W et al. *Ophthalmology* 2025. https://doi.org/10.1016/j.ophtha.2025.01.011

## Gene Signatures

In [ ]:
SIGNATURES = {

    # ---- RB tumour subtypes (Liu et al. 2021, Singh et al. 2018) ----
    "RB_subtype1_cone": [
        "ARR3", "RXRG", "OPN1SW", "OPN1MW", "THRB", "GNAT2",
        "PDE6H", "CNGB3", "CNGA3", "KCNV2", "RCVRN", "CRABP1",
    ],
    "RB_subtype2_stemness": [
        "VIM", "NES", "CD44", "SOX2", "PROM1", "FN1", "SNAI2",
        "TWIST1", "ZEB1", "MYCN", "HMGA1", "ID1", "ID3",
    ],

    # ---- Cone precursor CP4 invasive signature (Wan et al. 2025) ----
    "cone_precursor_CP4_invasive": [
        "SOX4", "MMP2", "MMP9", "TGFB1", "TGFB2", "TGFBR1",
        "SMAD3", "CTGF", "FN1", "CDH2",
    ],

    # ---- Cell cycle (Tirosh et al. 2016) ----
    "cell_cycle_S": [
        "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1",
        "UNG", "GINS2", "MCM6", "CDCA7", "DTL", "PRIM1", "UHRF1",
        "MLF1IP", "HELLS", "RFC2", "RPA2", "NASP", "RAD51AP1",
        "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7", "POLD3", "MSH2",
        "ATAD2", "RAD51", "RRM2", "CDC45", "CDC6", "EXO1", "TIPIN",
        "DSCC1", "BLM", "CASP8AP2", "USP1", "CLSPN", "POLA1", "CHAF1B",
        "BRIP1", "E2F8",
    ],
    "cell_cycle_G2M": [
        "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A",
        "NDC80", "CKS2", "NUF2", "CKS1B", "MKI67", "TMPO", "CENPF",
        "TACC3", "SMC4", "CCNB2", "CKAP2L", "CKAP2", "AURKB",
        "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1", "KIF20B", "HJURP",
        "CDCA3", "HN1", "CDC20", "TTK", "CDC25C", "KIF2C", "RANGAP1",
        "NCAPD2", "DLGAP5", "CDCA2", "CDCA8", "ECT2", "KIF23", "HMMR",
        "AURKA", "PSRC1", "ANLN", "LBR", "CKAP5", "CENPE", "CTCF",
        "NEK2", "G2E3", "GAS2L3", "CBX5", "CENPA",
    ],

    # ---- Hypoxia (HIF1A targets) ----
    "hypoxia_HIF1A": [
        "HIF1A", "VEGFA", "SLC2A1", "LDHA", "ENO1", "PGK1",
        "ALDOA", "CA9", "BNIP3", "BNIP3L", "LOX", "P4HA1",
    ],

    # ---- Retinal progenitor cells (RPC) ----
    "retinal_progenitor_RPC": [
        "VSX2", "PAX6", "SOX2", "LHX2", "ASCL1", "HES1", "DLL3",
        "NOTCH1", "PCNA", "TOP2A",
    ],

    # ---- TAM M1/M2 polarisation ----
    "TAM_M1": [
        "CD86", "IL1B", "IL6", "TNF", "CXCL10", "NOS2", "IRF5",
        "STAT1", "HLA-DRA", "CD64",
    ],
    "TAM_M2": [
        "CD163", "MRC1", "ARG1", "IL10", "TGFB1", "CCL18",
        "FOLR2", "LYVE1", "STAB1", "PDCD1LG2",
    ],
}

## Setup

In [ ]:
import warnings
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

ROOT     = Path("..").resolve()
IN_H5AD  = ROOT / "data" / "processed" / "06_cnv.h5ad"
OUT_H5AD = ROOT / "data" / "processed" / "07_subtyped.h5ad"
FIG_DIR  = ROOT / "results" / "figures"
TAB_DIR  = ROOT / "results" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Load CNV-Annotated Atlas

In [ ]:
print("Step 1 — Loading CNV-annotated atlas")
adata = sc.read_h5ad(IN_H5AD)
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"  obs columns: {list(adata.obs.columns)}")

## Step 2 — Score All Gene Signatures

Each signature score is stored as a column in `adata.obs`. The scores enable:
- Comparing subtype composition between intraocular and extraocular samples
- Continuous scoring rather than binary classification
- Cell cycle assignment for RNA velocity analysis

**Note:** `sc.tl.score_genes()` uses expression-bin-matched control gene sets, making scores robust to library-size effects (Tirosh et al. 2016).

In [ ]:
print("Step 2 — Scoring gene signatures")

for sig_name, genes in SIGNATURES.items():
    present = [g for g in genes if g in adata.var_names]
    absent  = [g for g in genes if g not in adata.var_names]
    if absent:
        print(f"  {sig_name}: {len(present)}/{len(genes)} genes found "
              f"(missing: {absent[:3]}{'...' if len(absent) > 3 else ''})")
    if len(present) < 3:
        print(f"  WARNING: {sig_name} has < 3 genes — score unreliable")
        adata.obs[f"score_{sig_name}"] = np.nan
        continue
    sc.tl.score_genes(
        adata,
        gene_list=present,
        score_name=f"score_{sig_name}",
        ctrl_size=50,
        n_bins=25,
        use_raw=False,
    )
    score_col = adata.obs[f"score_{sig_name}"]
    print(f"  {sig_name}: genes={len(present)}, "
          f"score range [{score_col.min():.3f}, {score_col.max():.3f}], "
          f"mean={score_col.mean():.3f}")

## Step 3 — Cell Cycle Phase Assignment

Cell cycle phase assignment is critical for RNA velocity: cycling cells produce unspliced mRNA that, without phase information, can be misinterpreted as directed transcriptional change.

- **S phase**: DNA synthesis markers (PCNA, MCM2, RRM1...)
- **G2M phase**: Mitosis markers (MKI67, TOP2A, CDK1...)
- **G1**: all other cells

Proliferating Subtype 2 cells are expected to be enriched in S/G2M.

In [ ]:
print("Step 3 — Cell cycle phase assignment")

s_genes   = [g for g in SIGNATURES["cell_cycle_S"]   if g in adata.var_names]
g2m_genes = [g for g in SIGNATURES["cell_cycle_G2M"] if g in adata.var_names]
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

phase_counts = adata.obs["phase"].value_counts()
print("  Cell cycle phase distribution:")
print(phase_counts.to_string())

## Step 4 — RB Subtype Assignment

For tumour cells (Cone_precursor, Tumour_proliferating), compute:
`subtype_diff = score_RB_subtype1_cone - score_RB_subtype2_stemness`

- **Positive (> 0.1)** → Subtype 1 (cone-like)
- **Negative (< -0.1)** → Subtype 2 (stemness)
- **Near zero** → Mixed / unresolved
- **Non-tumour cells** → labelled 'Non-tumour'

In [ ]:
print("Step 4 — RB subtype assignment")

tumour_types = {"Cone_precursor", "Tumour_proliferating"}
tumour_mask  = adata.obs["cell_type_broad"].isin(tumour_types)

if "cell_type_fine" in adata.obs.columns:
    fine_tumour = adata.obs["cell_type_fine"].str.startswith(("CP_", "TP_"), na=False)
    tumour_mask = tumour_mask | fine_tumour

subtype = pd.Series("Non-tumour", index=adata.obs_names)
s1 = adata.obs.get("score_RB_subtype1_cone",     pd.Series(np.nan, index=adata.obs_names))
s2 = adata.obs.get("score_RB_subtype2_stemness", pd.Series(np.nan, index=adata.obs_names))

diff = s1 - s2
subtype[tumour_mask & (diff >  0.1)] = "Subtype1_cone"
subtype[tumour_mask & (diff < -0.1)] = "Subtype2_stemness"
subtype[tumour_mask & (diff.abs() <= 0.1)] = "Mixed"

adata.obs["rb_subtype"] = subtype.astype("category")
counts = subtype[tumour_mask].value_counts()
print("  RB subtype assignment (tumour cells only):")
print(counts.to_string())

## Step 5 — Subtype Score UMAP

**What to look for:**
- `RB_subtype1_cone`: high scores in mature cone-like tumour clusters
- `RB_subtype2_stemness`: high scores in dedifferentiated / MYCN+ clusters
- `cone_precursor_CP4_invasive`: high scores in cells predicted to be invasive (Wan et al. 2025)
- `hypoxia_HIF1A`: potentially enriched in intraocular hypoxic core

In [ ]:
score_keys = [
    "score_RB_subtype1_cone",
    "score_RB_subtype2_stemness",
    "score_cone_precursor_CP4_invasive",
    "score_hypoxia_HIF1A",
    "rb_subtype",
    "phase",
]
present = [k for k in score_keys if k in adata.obs.columns]
ncols   = 3
nrows   = int(np.ceil(len(present) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
axes = axes.flatten() if nrows > 1 else axes.flatten()

for ax, key in zip(axes, present):
    sc.pl.umap(adata, color=key, ax=ax, show=False, frameon=False,
               size=2, alpha=0.7)
    ax.set_title(key.replace("score_", ""), fontsize=9)
for ax in axes[len(present):]:
    ax.set_visible(False)

plt.suptitle("Molecular subtype and signature scores", fontsize=12,
             fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / "subtype_scores_umap.pdf", dpi=200, bbox_inches="tight")
plt.show()

## Step 6 — Subtype Violin Plot by Dataset

**Key hypothesis:** Subtype 2 (stemness) scores should be higher in extraocular samples, reflecting the dedifferentiation that drives local extension.

In [ ]:
score_keys = ["score_RB_subtype1_cone", "score_RB_subtype2_stemness"]
present    = [k for k in score_keys if k in adata.obs.columns]
groupby    = "disease_stage" if "disease_stage" in adata.obs.columns else "dataset"

if present:
    fig, axes = plt.subplots(1, len(present), figsize=(7 * len(present), 5))
    if len(present) == 1:
        axes = [axes]
    for ax, key in zip(axes, present):
        sc.pl.violin(adata, keys=key, groupby=groupby, ax=ax, show=False, rotation=30)
        ax.set_title(key.replace("score_", ""), fontsize=10)
        ax.set_xlabel("")
    plt.suptitle(
        f"Subtype scores by {groupby}\n(Subtype 2 enrichment in extraocular expected)",
        fontsize=11, fontweight="bold",
    )
    plt.tight_layout()
    fig.savefig(FIG_DIR / "subtype_violin_by_dataset.pdf", dpi=200, bbox_inches="tight")
    plt.show()

## Step 7 — Subtype Composition per Sample

Stacked bar chart of RB subtype composition per sample (tumour cells only). Samples with more extraocular disease should show higher Subtype 2 fraction.

In [ ]:
if "rb_subtype" in adata.obs.columns:
    tumour_types = {"Cone_precursor", "Tumour_proliferating"}
    tumour_adata = adata[adata.obs["cell_type_broad"].isin(tumour_types)]

    if tumour_adata.n_obs > 0:
        comp = (
            tumour_adata.obs
            .groupby(["sample_id", "rb_subtype"])
            .size()
            .unstack(fill_value=0)
        )
        comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100

        colours = {
            "Subtype1_cone":     "#4C9BE8",
            "Subtype2_stemness": "#E84C4C",
            "Mixed":             "#F5A623",
            "Non-tumour":        "#AAAAAA",
        }
        cols = [c for c in colours if c in comp_pct.columns]

        fig, ax = plt.subplots(figsize=(max(8, len(comp_pct) * 0.8), 5))
        comp_pct[cols].plot(
            kind="bar", stacked=True, ax=ax,
            color=[colours[c] for c in cols], edgecolor="none",
        )
        ax.set_xlabel("Sample ID", fontsize=10)
        ax.set_ylabel("% tumour cells", fontsize=10)
        ax.set_title(
            "RB subtype composition per sample (tumour cells only)",
            fontsize=11, fontweight="bold",
        )
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        fig.savefig(FIG_DIR / "subtype_composition_bar.pdf", dpi=200, bbox_inches="tight")
        plt.show()

## Step 8 — Save Score Table and Atlas

In [ ]:
# Save score table
score_cols = [c for c in adata.obs.columns if c.startswith("score_")]
meta_cols  = ["sample_id", "dataset", "cell_type_broad", "rb_subtype",
              "phase", "cnv_load", "is_tumour_cnv"]
meta_cols  = [c for c in meta_cols if c in adata.obs.columns]
adata.obs[meta_cols + score_cols].to_csv(TAB_DIR / "subtype_scores_per_cell.csv")
print(f"Saved subtype scores -> {TAB_DIR / 'subtype_scores_per_cell.csv'}")

# Save atlas
print(f"\nSaving subtyped atlas -> {OUT_H5AD.name}")
adata.write_h5ad(OUT_H5AD, compression="gzip")
size_mb = OUT_H5AD.stat().st_size / 1e6
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes | {size_mb:.0f} MB")
print("  Added: .obs['score_*'], .obs['rb_subtype'], .obs['phase']")
print("  Next: Run notebook 07_rna_velocity.ipynb")